In [1]:
!pip install transformers
!pip install bitsandbytes
!pip install trl

In [39]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from peft import PeftModel
from tqdm import tqdm

In [3]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

In [4]:
print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
print("Токенизатор загружен!")

Загрузка токенизатора...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Токенизатор загружен!


In [28]:
print("Загрузка модели в 8-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)
print("Модель успешно загружена!")

Загрузка модели в 8-bit...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Модель успешно загружена!


In [6]:
test_text = "Привет, Квен!"
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens)

print(f"Успешно! Токенизатор работает.")
print(f"Исходный текст: {test_text}")
print(f"ID токенов в памяти: {tokens}")
print(f"Декодированный текст: {decoded}")

Успешно! Токенизатор работает.
Исходный текст: Привет, Квен!
ID токенов в памяти: [53645, 26991, 8178, 11, 35379, 131423, 0]
Декодированный текст: Привет, Квен!


In [7]:
# Задаем шаблон диалога
prompt = "Скажи hello world!"
messages = [
    {
        "role": "system",
        "content": "Ты — продвинутый ИИ-ассистент. Отвечаешь четко, развернуто и только на русском языке."
    },
    {
        "role": "user",
        "content": prompt
    }
]

# Применяем чат-шаблон Qwen (добавляет специальные токены разметки диалога)
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Переводим текст в тензоры для модели
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)



In [8]:
print("Генерация ответа...\n" + "="*40)

# Запуск генерации
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=128,   # Максимальная длина ответа
    temperature=0.7,      # Креативность (0.7 — оптимально для баланса точности и вариативности)
    do_sample=True,
    top_p=0.9             # Ограничение выборки маловероятных токенов
)

# Вырезаем из ответа входной промпт, чтобы остался только чистый текст модели
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Декодируем токены обратно в понятный текст
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)
print("="*40)

Генерация ответа...
Hello World!


In [16]:
dataset = load_dataset("d0rj/alpaca-cleaned-ru", split="train[:5000]")
DATASET_CHAT_COLUMN = "chat"

In [29]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./results_fine_tuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=5,
    save_strategy="steps",
    save_steps=50,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    dataset_text_field="text" # Explicitly set dataset_text_field to 'text'
)

In [30]:
def formatting_function(example):
    # Construct the messages list in the chat format from 'instruction', 'input', and 'output'
    messages = []
    if example.get("instruction"): # 'instruction' is the user's main query
        user_content = example["instruction"]
        if example.get("input"): # 'input' provides additional context for the instruction
            user_content += f"\n{example['input']}"
        messages.append({"role": "user", "content": user_content})

    if example.get("output"): # 'output' is the assistant's response
        messages.append({"role": "assistant", "content": example["output"]})

    # Apply the chat template if messages are present and return the string directly
    if messages:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return "" # Return an empty string if no messages can be formed for this example

# Remove training_args.dataset_text_field as formatting_func handles the conversion
# training_args.dataset_text_field = DATASET_CHAT_COLUMN # This line is removed

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
    formatting_func=formatting_function
)

In [31]:
print("Начало обучения модели...")
trainer.train()
print("Модель обучена!")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Начало обучения модели...


Step,Training Loss


TrainOutput(global_step=5, training_loss=1.6454866409301758, metrics={'train_runtime': 96.0015, 'train_samples_per_second': 0.417, 'train_steps_per_second': 0.052, 'total_flos': 136367322906624.0, 'train_loss': 1.6454866409301758, 'entropy': 1.2725057482719422, 'num_tokens': 11388.0, 'mean_token_accuracy': 0.6681533142924309, 'epoch': 0.008})

In [32]:
print("Сохранение обученных LoRA весов...")
trainer.model.save_pretrained("my_custom_chat_bot")
tokenizer.save_pretrained("my_custom_chat_bot")
print("Все готово!")

Сохранение обученных LoRA весов...
Все готово!


In [35]:
lora_weights_path = "my_custom_chat_bot" # путь к вашей сохраненной папке

# Общая конфигурация квантования
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Загрузка БАЗОВОЙ модели...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Загрузка ДООБУЧЕННОЙ модели (База + LoRA)...")
# Сначала берем ту же базу
finetuned_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
# И подгружаем в нее ваши обученные адаптеры
finetuned_model = PeftModel.from_pretrained(finetuned_model, lora_weights_path)

Загрузка токенизатора...
Загрузка БАЗОВОЙ модели...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Загрузка ДООБУЧЕННОЙ модели (База + LoRA)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [36]:
print("Загрузка тестовой выборки...")
# 1. Берем 500 примеров, которые строго идут ПОСЛЕ обучающей выборки
test_dataset = load_dataset("d0rj/alpaca-cleaned-ru", split="train[5000:5500]")

# 2. Модифицируем вашу функцию форматирования, чтобы Hugging Face datasets
# мог применить её через метод .map() к каждой строке
def apply_formatting_to_dataset(example):
    # Ваша логика форматирования
    messages = []
    if example.get("instruction"):
        user_content = example["instruction"]
        if example.get("input"):
            user_content += f"\n{example['input']}"
        messages.append({"role": "user", "content": user_content})

    if example.get("output"):
        messages.append({"role": "assistant", "content": example["output"]})

    if messages:
        # Важно: .map() ожидает, что функция вернет словарь с новой колонкой
        return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}
    else:
        return {"text": ""}

# 3. Трансформируем тестовый датасет
test_dataset = test_dataset.map(apply_formatting_to_dataset)

# На всякий случай отфильтруем пустые строки, если они вдруг появились
test_dataset = test_dataset.filter(lambda x: x["text"] != "")

print(f"Успешно подготовлено тестовых строк: {len(test_dataset)}")

Загрузка тестовой выборки...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Успешно подготовлено тестовых строк: 500


In [41]:
# Обязательно определяем устройство, на котором будут идти расчеты
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Расчеты будут производиться на: {device}")

def calculate_perplexity(model, tokenizer, dataset, max_length=1024):
    model.eval()
    nlls = []

    print("Расчет перплексии...")
    with torch.no_grad():
        for example in tqdm(dataset):
            # Токенизируем уже готовый отформатированный текст
            encodings = tokenizer(example['text'], return_tensors="pt", truncation=True, max_length=max_length)
            input_ids = encodings.input_ids.to(device)
            target_ids = input_ids.clone()

            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss
            nlls.append(neg_log_likelihood)

    ppl = torch.exp(torch.stack(nlls).mean()).item()
    return ppl

# Запуск подсчета
base_ppl = calculate_perplexity(base_model, tokenizer, test_dataset)
finetuned_ppl = calculate_perplexity(finetuned_model, tokenizer, test_dataset)

print("\n=== РЕЗУЛЬТАТЫ СРАВНЕНИЯ ПЕРПЛЕКСИИ ===")
print(f"Базовая модель PPL:    {base_ppl:.4f}")
print(f"Дообученная модель PPL: {finetuned_ppl:.4f}")
print(f"Разница: {base_ppl - finetuned_ppl:.4f} (Чем ниже PPL, тем модель лучше)")

Расчеты будут производиться на: cuda
Расчет перплексии...


100%|██████████| 500/500 [02:02<00:00,  4.08it/s]


Расчет перплексии...


100%|██████████| 500/500 [02:22<00:00,  3.50it/s]


=== РЕЗУЛЬТАТЫ СРАВНЕНИЯ ПЕРПЛЕКСИИ ===
Базовая модель PPL:    12.9533
Дообученная модель PPL: 5.1732
Разница: 7.7801 (Чем ниже PPL, тем модель лучше)
